## Statsmodels

Agencia: female 0.965 vs. male 0.961
Hartmann: female más surprise (14.2% vs. 12.5%)
VADER: female compound 0.059 vs. male 0.042

Pero ninguna de esas diferencias viene acompañada de una respuesta a la pregunta más importante: ¿esa diferencia es real o podría ser casualidad?
statsmodels y scipy responden esa pregunta mediante tests estadísticos. El resultado es un p-valor:


> p < 0.05 → la diferencia es estadísticamente significativa, es decir, muy probablemente real

> p > 0.05 → la diferencia podría ser fruto del azar


Los tests que usaremos:

· Mann-Whitney U — compara dos grupos (female vs. male) en una variable continua como el compound de VADER o el ratio de agencia. Se usa en lugar de una t-test porque los datos de NLP no siguen distribución normal.

· Kruskal-Wallis — igual que Mann-Whitney pero para tres grupos simultáneamente (Best Picture vs. Original Screenplay vs. Adapted Screenplay).

· Correlación de Spearman — para la dimensión temporal: ¿ha cambiado el perfil emocional entre 2000 y 2025?

In [3]:
import pickle
import pandas as pd
import numpy as np
from scipy import stats

# Cargar agencia desde CSV
df_agencia = pd.read_csv("resultados_agencia_filtrado.csv")

# Cargar hartmann y vader desde pickle
df_hartmann = pd.read_pickle("resultados_hartmann_frases.pkl")
df_vader = pd.read_pickle("resultados_vader.pkl")

print(f"Agencia: {len(df_agencia)} personajes")
print(f"Hartmann: {len(df_hartmann)} frases")
print(f"VADER: {len(df_vader)} frases")

Agencia: 1380 personajes
Hartmann: 91236 frases
VADER: 70255 frases


In [4]:
# TEST 1 — Mann-Whitney U: agencia female vs. male
female_agencia = df_agencia[df_agencia['genero'] == 'female']['agencia']
male_agencia = df_agencia[df_agencia['genero'] == 'male']['agencia']

stat, p = stats.mannwhitneyu(female_agencia, male_agencia, alternative='two-sided')

print("=== AGENCIA NARRATIVA ===")
print(f"Female — media: {female_agencia.mean():.3f}, n={len(female_agencia)}")
print(f"Male   — media: {male_agencia.mean():.3f}, n={len(male_agencia)}")
print(f"Mann-Whitney U: {stat:.1f}, p-valor: {p:.4f}")
if p < 0.05:
    print("→ Diferencia estadísticamente significativa (p < 0.05)")
else:
    print("→ Diferencia NO estadísticamente significativa (p > 0.05)")

=== AGENCIA NARRATIVA ===
Female — media: 0.964, n=394
Male   — media: 0.960, n=986
Mann-Whitney U: 201078.0, p-valor: 0.2937
→ Diferencia NO estadísticamente significativa (p > 0.05)


**Interpretación**

p = 0.2937 → diferencia NO significativa

La diferencia de 0.004 puntos entre female (0.964) y male (0.960) en agencia narrativa no es estadísticamente distinguible del azar. Con 75 películas y esta variabilidad, no podemos afirmar que existe una diferencia real en agencia entre géneros.

In [5]:
# TEST 2 — Mann-Whitney U: emociones Hartmann female vs. male
print("=== HARTMANN — TESTS POR EMOCIÓN ===\n")

emociones = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

for emocion in emociones:
    female_vals = (df_hartmann[df_hartmann['genero'] == 'female']['emocion'] == emocion).astype(int)
    male_vals = (df_hartmann[df_hartmann['genero'] == 'male']['emocion'] == emocion).astype(int)
    stat, p = stats.mannwhitneyu(female_vals, male_vals, alternative='two-sided')
    significativo = "✓ significativo" if p < 0.05 else "✗ no significativo"
    print(f"{emocion:<10} p={p:.4f}  {significativo}")

=== HARTMANN — TESTS POR EMOCIÓN ===

anger      p=0.3374  ✗ no significativo
disgust    p=0.4234  ✗ no significativo
fear       p=0.1514  ✗ no significativo
joy        p=0.0000  ✓ significativo
neutral    p=0.0000  ✓ significativo
sadness    p=0.0000  ✓ significativo
surprise   p=0.0000  ✓ significativo


**Interpretación**

Las diferencias significativas son joy, neutral, sadness y surprise. Estas cuatro emociones tienen diferencias reales entre géneros que no son fruto del azar.

Anger, disgust y fear no son significativas — las diferencias que veíamos en los porcentajes eran demasiado pequeñas para ser estadísticamente defendibles.

In [6]:
# TEST 3 — Mann-Whitney U: compound VADER female vs. male
female_compound = df_vader[df_vader['genero'] == 'female']['compound']
male_compound = df_vader[df_vader['genero'] == 'male']['compound']

stat, p = stats.mannwhitneyu(female_compound, male_compound, alternative='two-sided')

print("=== VADER — COMPOUND SCORE ===")
print(f"Female — media: {female_compound.mean():.3f}, n={len(female_compound)}")
print(f"Male   — media: {male_compound.mean():.3f}, n={len(male_compound)}")
print(f"Mann-Whitney U: {stat:.1f}, p-valor: {p:.4f}")
if p < 0.05:
    print("→ Diferencia estadísticamente significativa (p < 0.05)")
else:
    print("→ Diferencia NO estadísticamente significativa (p > 0.05)")

=== VADER — COMPOUND SCORE ===
Female — media: 0.059, n=19077
Male   — media: 0.042, n=51178
Mann-Whitney U: 499039171.5, p-valor: 0.0000
→ Diferencia estadísticamente significativa (p < 0.05)


**Interpretación**

Perfecto. p=0.0000 — diferencia altamente significativa.

Female compound: 0.059 vs. Male compound: 0.042 — diferencia significativa (p < 0.001)

El sentimiento femenino es significativamente más positivo que el masculino. Con 70.000 frases analizadas la diferencia es pequeña en magnitud pero estadísticamente robusta.

In [8]:
# TEST 4 — Kruskal-Wallis completo por award
print("=== KRUSKAL-WALLIS — DIFERENCIAS POR AWARD ===\n")

awards = df_hartmann['award'].unique()

print("--- HARTMANN: todas las emociones por award ---")
emociones = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
for emocion in emociones:
    grupos = [df_hartmann[df_hartmann['award'] == a]['emocion'].eq(emocion).astype(int) for a in awards]
    stat, p = stats.kruskal(*grupos)
    significativo = "✓ significativo" if p < 0.05 else "✗ no significativo"
    print(f"{emocion:<10} H={stat:.3f}, p={p:.4f}  {significativo}")

print("\n--- VADER: compound por award ---")
grupos_vader = [df_vader[df_vader['award'] == a]['compound'] for a in awards]
stat, p = stats.kruskal(*grupos_vader)
print(f"compound   H={stat:.3f}, p={p:.4f}  {'✓ significativo' if p < 0.05 else '✗ no significativo'}")

print("\n--- AGENCIA: por award ---")
grupos_agencia = [df_agencia[df_agencia['premio'] == a]['agencia'] for a in df_agencia['premio'].unique()]
stat, p = stats.kruskal(*grupos_agencia)
print(f"agencia    H={stat:.3f}, p={p:.4f}  {'✓ significativo' if p < 0.05 else '✗ no significativo'}")

=== KRUSKAL-WALLIS — DIFERENCIAS POR AWARD ===

--- HARTMANN: todas las emociones por award ---
anger      H=54.164, p=0.0000  ✓ significativo
disgust    H=0.657, p=0.7200  ✗ no significativo
fear       H=7.816, p=0.0201  ✓ significativo
joy        H=81.830, p=0.0000  ✓ significativo
neutral    H=69.977, p=0.0000  ✓ significativo
sadness    H=3.005, p=0.2226  ✗ no significativo
surprise   H=24.908, p=0.0000  ✓ significativo

--- VADER: compound por award ---
compound   H=69.373, p=0.0000  ✓ significativo

--- AGENCIA: por award ---
agencia    H=1.072, p=0.5851  ✗ no significativo


el test Kruskal-Wallis compara los tres awards entre sí pero el output solo muestra si hay diferencia significativa entre ellos, no cuál award difiere de cuál.
Lo que nos dice el output es:

· anger, fear, joy, neutral, surprise → hay diferencias significativas entre las tres categorías de premio

· disgust, sadness → no hay diferencias significativas entre awards

· VADER compound → diferencias significativas entre awards

· agencia → no hay diferencias significativas entre awards

In [9]:
from scipy.stats import mannwhitneyu

print("=== POST-HOC: qué awards difieren entre sí ===\n")

awards = ['Best Picture', 'Original Screenplay', 'Adapted Screenplay']
pares = [
    ('Best Picture', 'Original Screenplay'),
    ('Best Picture', 'Adapted Screenplay'),
    ('Original Screenplay', 'Adapted Screenplay')
]

print("--- HARTMANN: joy ---")
for a1, a2 in pares:
    g1 = df_hartmann[df_hartmann['award'] == a1]['emocion'].eq('joy').astype(int)
    g2 = df_hartmann[df_hartmann['award'] == a2]['emocion'].eq('joy').astype(int)
    stat, p = mannwhitneyu(g1, g2, alternative='two-sided')
    significativo = "✓" if p < 0.05 else "✗"
    print(f"  {a1} vs {a2}: p={p:.4f} {significativo}")

print("\n--- VADER: compound ---")
for a1, a2 in pares:
    g1 = df_vader[df_vader['award'] == a1]['compound']
    g2 = df_vader[df_vader['award'] == a2]['compound']
    stat, p = mannwhitneyu(g1, g2, alternative='two-sided')
    significativo = "✓" if p < 0.05 else "✗"
    print(f"  {a1} vs {a2}: p={p:.4f} {significativo}")

=== POST-HOC: qué awards difieren entre sí ===

--- HARTMANN: joy ---
  Best Picture vs Original Screenplay: p=0.0000 ✓
  Best Picture vs Adapted Screenplay: p=0.6683 ✗
  Original Screenplay vs Adapted Screenplay: p=0.0000 ✓

--- VADER: compound ---
  Best Picture vs Original Screenplay: p=0.0000 ✓
  Best Picture vs Adapted Screenplay: p=0.0010 ✓
  Original Screenplay vs Adapted Screenplay: p=0.0000 ✓


In [15]:
# Correlación de Spearman
# VADER compound por año y género
for genero in ['female', 'male']:
    datos = df_vader[df_vader['genero'] == genero].groupby('oscar_year')['compound'].mean()
    corr, p = stats.spearmanr(datos.index, datos.values)
    significativo = "✓ significativo" if p < 0.05 else "✗ no significativo"
    print(f"VADER compound {genero}: r={corr:.3f}, p={p:.4f}  {significativo}")

print()

# Hartmann joy por año y género
for genero in ['female', 'male']:
    datos = df_hartmann[df_hartmann['genero'] == genero].groupby('oscar_year')['emocion'].apply(
        lambda x: (x == 'joy').mean() * 100)
    corr, p = stats.spearmanr(datos.index, datos.values)
    significativo = "✓ significativo" if p < 0.05 else "✗ no significativo"
    print(f"Hartmann joy {genero}: r={corr:.3f}, p={p:.4f}  {significativo}")

print()

# Agencia por año — no tiene columna de año, saltamos este test
print("Agencia temporal: no disponible (columna de año no incluida en el CSV)")

VADER compound female: r=-0.085, p=0.6793  ✗ no significativo
VADER compound male: r=0.065, p=0.7513  ✗ no significativo

Hartmann joy female: r=-0.021, p=0.9195  ✗ no significativo
Hartmann joy male: r=0.069, p=0.7362  ✗ no significativo

Agencia temporal: no disponible (columna de año no incluida en el CSV)


Los valores de r son muy cercanos a 0 y los p-valores muy altos. No hay ninguna tendencia temporal — ni de mejora ni de empeoramiento — en el perfil emocional del cine premiado entre 2000 y 2025.